# Main Results

This notebook loads the precomputed suite outputs in `results/multi_regime_suite/` and recreates a small set of summary views for thesis writing and sanity checks.

In [ ]:
from __future__ import annotations

import csv
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
RESULTS = ROOT / 'results' / 'multi_regime_suite'
AGGREGATED = RESULTS / 'suite_aggregated.csv'
RUNS = RESULTS / 'suite_runs.csv'

with AGGREGATED.open() as handle:
    aggregated_rows = list(csv.DictReader(handle))

with RUNS.open() as handle:
    run_rows = list(csv.DictReader(handle))

print(f'Loaded {len(aggregated_rows)} aggregated rows and {len(run_rows)} per-run rows from {RESULTS}')

In [ ]:
regime_rows = [row for row in aggregated_rows if row['study_name'] == 'regime']
grouped = defaultdict(list)
for row in regime_rows:
    grouped[row['method']].append(row)

plt.figure(figsize=(10, 4))
for method, rows in sorted(grouped.items()):
    xs = [row['study_value'] for row in rows]
    ys = [float(row['mean_best_raw_objective']) for row in rows]
    plt.plot(xs, ys, marker='o', label=method)
plt.xticks(rotation=20)
plt.ylabel('Mean best raw objective')
plt.title('Optimizer performance by regime')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_by_study = {}
for row in aggregated_rows:
    key = (row['study_name'], row['study_value'])
    score = float(row['mean_best_raw_objective'])
    current = best_by_study.get(key)
    if current is None or score < current[1]:
        best_by_study[key] = (row['method'], score)

for (study_name, study_value), (method, score) in sorted(best_by_study.items()):
    print(f'{study_name:14s} {study_value:>12}: {method:20s} {score: .4f}')